# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a step-by-step guide for loading and exploring the FAIR^2 dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

## Dataset Source
The dataset is available through a Croissant schema at:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`


In [ ]:
# Make sure `mlcroissant` and dependencies are installed
!pip install -U mlcroissant pandas

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the URL for the Croissant schema
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Show the dataset metadata
metadata = dataset.metadata
print("Dataset title: ", getattr(metadata, 'name', ''))
print("Description: ", getattr(metadata, 'description', ''))
print("Identifier: ", getattr(metadata, 'identifier', ''))
print("Date published: ", getattr(metadata, 'datePublished', ''))
print("Citation string: ", getattr(metadata, 'citeAs', ''))

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List all available RecordSets in the Croissant package by their @id and see their fields

print('RecordSets in dataset:')
if hasattr(metadata, "recordSet") and metadata.recordSet:
    record_sets = metadata.recordSet
else:
    # If not present directly (which can happen if declared elsewhere), fall back to searching the internal dict
    # Try to use internal objects in dataset
    record_sets = []
    if hasattr(dataset, "graph"):
        for value in dataset.graph.values():
            if isinstance(value, mlc.croissant.RecordSet):
                record_sets.append(value)
    if not record_sets:
        # As fallback, use dataset._graph (internal, not public API)
        for obj in dataset._graph:
            if getattr(obj, '@type', None) in ('cr:RecordSet', 'RecordSet'):
                record_sets.append(obj)

for rs in record_sets:
    print(f"\nRecordSet @id: {getattr(rs, '@id', '')}")
    print(f"  Name: {getattr(rs, 'name', '')}")
    # List fields/columns
    field_ids = []
    if hasattr(rs, "field") and rs.field:
        field_ids = rs.field
    elif hasattr(rs, "column") and rs.column:
        field_ids = rs.column
    if field_ids:
        print("  Fields/columns (@id):")
        for fid in field_ids:
            print(f"    - {getattr(fid, '@id', fid)}")
    else:
        print("  No fields/columns registered.")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. We use the record set and field `@id`s from above.

In [ ]:
# Extract all available record sets by their @id
# (You may adjust or select specific ones by @id once you see the list above)

# Reuse extraction method for backward compatibility
def get_recordset_ids(record_sets):
    ids = []
    for rs in record_sets:
        rsid = getattr(rs, '@id', None)
        if rsid:
            ids.append(rsid)
    return ids

record_set_ids = get_recordset_ids(record_sets)

if not record_set_ids:
    raise Exception("No RecordSets found to extract from; please check the record sets displayed above.")

dataframes = {}
for rsid in record_set_ids:
    print(f"Loading data from RecordSet: {rsid}")
    records_iter = dataset.records(record_set=rsid)
    records = list(records_iter)
    dataframes[rsid] = pd.DataFrame(records)
    print(f"{len(dataframes[rsid])} records loaded. Columns: {list(dataframes[rsid].columns)}\n")

# We'll pick the first RecordSet (usually the main tabular data) for demonstration
main_record_set_id = record_set_ids[0]
print(f"Main RecordSet id: {main_record_set_id}")
print(f"Columns: {list(dataframes[main_record_set_id].columns)}")
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)

Let's perform common data processing steps:
- Filtering records by a numeric column,
- Normalizing a numeric field,
- Grouping by a categorical field, etc.

Make sure to use each field/column's `@id` for selection.

In [ ]:
# Inspect numeric columns by their names (which are usually their field @id or labels)
df = dataframes[main_record_set_id]
print("First five records:")
display(df.head())

# Try to select a numeric field. We'll heuristically pick the first numeric-like column.
numeric_field_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col]) or (df[col].dtype == object and df[col].str.replace(r'[^\d.]','',regex=True).str.len().gt(0).any())]

# If columns with names like 'age' exist, prefer them
prefer = ['age', 'Age', 'interval', 'Interval', 'diagnosisInterval', 'diagnosis_interval', 'IntervalMonths']
selected_numeric = None
for c in prefer:
    if c in df.columns:
        selected_numeric = c
        break
if not selected_numeric and numeric_field_candidates:
    selected_numeric = numeric_field_candidates[0]

print(f"Selected numeric field for analysis: {selected_numeric}")
# Clean/convert the numeric field if needed
numeric_field = selected_numeric
df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')

# Filter: e.g., only those over a threshold
threshold = df[numeric_field].quantile(0.5) # Use median as threshold
filtered_df = df[df[numeric_field] > threshold].copy()
print(f"Filtered records where {numeric_field} > median ({threshold:.2f}): {len(filtered_df)} rows")
display(filtered_df.head())

# Normalize
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try grouping by a likely categorical column (e.g. 'sex', 'Sex', 'msi_status', etc.)
group_candidates = ['sex', 'Sex', 'msi_status', 'MSI_status', 'group', 'anatomical_location', 'location', 'Site']
group_field = None
for gc in group_candidates:
    if gc in df.columns:
        group_field = gc
        break
if not group_field:
    for col in df.columns:
        if df[col].dtype == object and df[col].nunique() < 10:
            group_field = col
            break

print(f"Grouping by: {group_field}")
if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
    grouped_df.columns = [f"mean_{numeric_field}"]
    display(grouped_df)
else:
    print("No suitable group field found.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 5))
sns.histplot(df[numeric_field].dropna(), kde=True, bins=15)
plt.title(f'Distribution of {numeric_field}')
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

if group_field:
    plt.figure(figsize=(10, 5))
    sns.boxplot(data=df, x=group_field, y=numeric_field)
    plt.title(f'{numeric_field} by {group_field}')
    plt.ylabel(numeric_field)
    plt.xlabel(group_field)
    plt.show()

## 6. Conclusion

- Using the Croissant-standardized FAIR² dataset, we've loaded the metadata and records programmatically.
- We've explored available record sets, fields, and their `@id`s, and loaded data into Pandas DataFrames.
- Performed basic EDA: filtered, normalized, and grouped numeric attributes; visualized distributions;
- All data access (especially record sets and fields) was handled using their unique Croissant `@id` identifiers per best practices.

This workflow can be adapted for any Croissant dataset and extended for more advanced ML/data science analyses.